<img src="../../assets/sambanova_logo.png" alt="SambaNova" width="250">

# Session 1: Build Your First Agent

**Deep Agents from Scratch** — Data Science Dojo x SambaNova

---

In this notebook, we'll build a **ReAct agent** from the ground up using LangGraph and SambaNova's MiniMax-M2.5 model.

**What we'll build:**
- A simple LLM chain (baseline — no tools, no planning, no iteration)
- A ReAct agent that can **plan**, **search the web**, **write files**, and **iterate on quality**

**The task:** *"Research a topic, create a plan, and write a summary report."*

By the end, you'll see the dramatic difference between a static chain and a dynamic agent.

In [ ]:
import os
import sys
from dotenv import load_dotenv

load_dotenv(os.path.join("..", "..", ".env"), override=True)

# Add parent notebooks dir to path for utils import
sys.path.insert(0, os.path.join("..", "."))

%load_ext autoreload
%autoreload 2

In [ ]:
# Validate required API keys
required_keys = ["SAMBANOVA_API_KEY", "TAVILY_API_KEY"]
missing = [k for k in required_keys if not os.environ.get(k)]
if missing:
    raise EnvironmentError(
        f"Missing required API keys: {', '.join(missing)}\n"
        f"Copy .env.example to .env and fill in your keys:\n"
        f"  cp .env.example .env"
    )
print("All required API keys present.")

### Langfuse Observability (Optional)

If you have [Langfuse](https://langfuse.com) credentials, we'll enable tracing so you can inspect every LLM call and tool invocation in the Langfuse dashboard. If not, the notebook works fine without it.

In [ ]:
from langfuse.langchain import CallbackHandler

langfuse_handler = None
if os.environ.get("LANGFUSE_PUBLIC_KEY") and os.environ.get("LANGFUSE_SECRET_KEY"):
    langfuse_handler = CallbackHandler()
    print("Langfuse tracing enabled — traces will appear in your Langfuse dashboard.")
else:
    print("Langfuse keys not found — tracing disabled. Set LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY to enable.")


def get_config():
    """Build config with Langfuse callback when available."""
    if langfuse_handler:
        return {"callbacks": [langfuse_handler]}
    return {}

## What is a ReAct Agent?

**ReAct** = **Re**ason + **Act**

A ReAct agent follows an iterative loop:

1. **Observe** — Receive input or tool results
2. **Think** — Reason about what to do next
3. **Act** — Call a tool or produce a final answer
4. **Repeat** — Continue until the task is complete

This is what separates an *agent* from a simple LLM call: agents **plan**, **use tools**, and **iterate** to solve complex problems.

---

## Part 1: The Simple Chain (Baseline)

Let's start with a basic LLM call — no tools, no agent loop. Just ask the model directly and see what happens.

In [ ]:
from langchain_sambanova import ChatSambaNova

llm = ChatSambaNova(model="MiniMax-M2.5", temperature=0.0)

# Simple chain: just ask the LLM directly
response = llm.invoke(
    "Research the latest developments in AI agents, create a plan, and write a summary report.",
    config=get_config(),
)
print(response.content)

> **Notice:** The chain produces a generic response from training data. No real research, no tool use, no iteration. It can't search the web, it can't save its work to files, and it can't create a structured plan. Everything comes from memory alone.

---

## Part 2: Adding Tools

Tools give agents **capabilities**. We'll define three tools that enable the full workflow:

| Tool | Purpose |
|------|--------|
| `web_search` | Search the web for current information |
| `write_todos` | Create a structured plan with tasks |
| `write_file` | Save research and reports to files |

Together these let the agent **plan** → **search** → **write** → **iterate**.

### Custom Agent State

Before defining tools, we need a custom state that can hold our TODO list and virtual file system. This extends `AgentState` with two new fields.

In [ ]:
from typing import Annotated, Literal, NotRequired
from typing_extensions import TypedDict
from langchain.agents import AgentState


class Todo(TypedDict):
    """A structured task item for tracking progress."""
    content: str
    status: Literal["pending", "in_progress", "completed"]


def file_reducer(left, right):
    """Merge file dicts — new files override existing ones."""
    if left is None:
        return right
    elif right is None:
        return left
    return {**left, **right}


class DeepAgentState(AgentState):
    """Agent state with TODO tracking and a virtual file system."""
    todos: NotRequired[list[Todo]]
    files: Annotated[NotRequired[dict[str, str]], file_reducer]


print("State schema defined: DeepAgentState with todos + files")

### Tool Definitions

Now we define the three tools. Notice that `write_todos` and `write_file` return a `Command` instead of a string — this lets them **update agent state directly**. This is how tools write back to the agent's memory.

In [ ]:
from langchain_core.tools import tool, InjectedToolCallId
from langchain_core.messages import ToolMessage
from langgraph.prebuilt import InjectedState
from langgraph.types import Command


@tool
def web_search(query: str) -> str:
    """Search the web for current information.

    Args:
        query: The search query string.
    """
    from tavily import TavilyClient

    client = TavilyClient()
    results = client.search(query, max_results=3)
    formatted = []
    for r in results.get("results", []):
        formatted.append(f"**{r['title']}**\n{r['content']}\nSource: {r['url']}")
    return "\n\n---\n\n".join(formatted) if formatted else "No results found."


@tool(parse_docstring=True)
def write_todos(
    todos: list[Todo],
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> Command:
    """Create or update the agent's TODO list for task planning and tracking.

    Use this to break complex tasks into steps and track progress.
    Only one task should be in_progress at a time.

    Args:
        todos: List of Todo items with content and status.
        tool_call_id: Tool call identifier (injected).
    """
    return Command(
        update={
            "todos": todos,
            "messages": [
                ToolMessage(f"Updated todo list: {len(todos)} items", tool_call_id=tool_call_id)
            ],
        }
    )


@tool(parse_docstring=True)
def write_file(
    file_path: str,
    content: str,
    state: Annotated[dict, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> Command:
    """Write content to a file in the virtual filesystem.

    Use this to save research findings, reports, or any artifacts.
    Files persist in agent state across the entire conversation.

    Args:
        file_path: Path where the file should be created/updated.
        content: Content to write to the file.
        state: Agent state (injected).
        tool_call_id: Tool call identifier (injected).
    """
    files = state.get("files", {}) or {}
    files[file_path] = content
    return Command(
        update={
            "files": files,
            "messages": [
                ToolMessage(f"Updated file: {file_path}", tool_call_id=tool_call_id)
            ],
        }
    )


tools = [web_search, write_todos, write_file]
print(f"Tools defined: {[t.name for t in tools]}")

---

## Part 3: Build the ReAct Agent

Now we wire everything together using `create_agent`:
- **Model**: SambaNova MiniMax-M2.5
- **Tools**: Web Search + TODO Planning + File Writing
- **State**: Custom `DeepAgentState` with todos and files
- **Prompt**: Instructions that guide the agent to plan, research, and write

The agent will automatically handle the ReAct loop — reasoning about which tools to call and when to stop.

In [ ]:
from langchain.agents import create_agent
from IPython.display import Image, display

SYSTEM_PROMPT = """You are a helpful research assistant with access to tools for planning, searching, and writing.

When given a research task, follow this workflow:
1. **Plan**: Use write_todos to create a structured task list
2. **Research**: Use web_search to find current, accurate information
3. **Write**: Use write_file to save your findings and final report
4. **Iterate**: Update your todo list as you complete each step

Always start by creating a plan. Save your final report to a file."""

model = ChatSambaNova(model="MiniMax-M2.5", temperature=0.0)

agent = create_agent(
    model,
    tools,
    system_prompt=SYSTEM_PROMPT,
    state_schema=DeepAgentState,
)

# Visualize the agent graph
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

---

## Part 4: Agent vs Chain — The Same Task

Let's give the **agent** the exact same task the simple chain struggled with. Watch how the agent:
- **Plans** with a TODO list
- **Searches** the web for real information
- **Writes** its report to a file
- **Iterates** by updating its plan as it goes

In [ ]:
from utils import format_messages

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Research the latest developments in AI agents, create a plan, and write a summary report.",
            }
        ],
    },
    config=get_config(),
)

format_messages(result["messages"])

### Inspecting Agent State

Unlike a simple chain, our agent maintained structured state throughout. Let's look at what it produced:

In [ ]:
# Inspect the TODO list
print("TODO List:")
print("=" * 50)
status_emoji = {"pending": "\u23f3", "in_progress": "\U0001f504", "completed": "\u2705"}
for todo in result.get("todos", []):
    emoji = status_emoji.get(todo["status"], "?")
    print(f"  {emoji} {todo['content']} ({todo['status']})")

# Inspect files written
print(f"\nFiles Written:")
print("=" * 50)
for file_path, content in result.get("files", {}).items():
    print(f"\n--- {file_path} ({len(content)} chars) ---")
    print(content[:1000])
    if len(content) > 1000:
        print("...[truncated]")

---

### The Difference

Compare the agent's response to the simple chain above:

| | Simple Chain | Agent with Tools |
|---|---|---|
| **Planning** | None | Created a TODO list and tracked progress |
| **Research** | Training data only | Searched the web for current information |
| **Output** | Text in chat | Wrote a report to a file |
| **Iteration** | Single pass | Multiple tool calls, updated plan as it went |

This is what "deep" means: **planning, acting, observing, iterating**.

---

## Part 5: Breaking Down What We Built

Let's understand the key concepts that made this agent work.

### 1. State — The Agent's Memory

```python
class DeepAgentState(AgentState):
    todos: list[Todo]       # Structured task tracking
    files: dict[str, str]   # Virtual file system
```

State is what separates an agent from a chain. The agent can **write to** and **read from** state across multiple steps. This is how it maintains context, tracks progress, and builds artifacts over time.

### 2. Tools with `Command` — Writing Back to State

Normal tools return a string. But `write_todos` and `write_file` return a `Command` — this lets them **update agent state directly**:

```python
return Command(
    update={
        "files": files,                    # Update the virtual filesystem
        "messages": [ToolMessage(...)],     # Send confirmation back to the agent
    }
)
```

This is a powerful pattern — tools don't just return information, they **change the world** the agent operates in.

### 3. The ReAct Loop — Plan, Act, Observe, Repeat

The agent autonomously decided to:
1. Create a plan (write_todos)
2. Search for information (web_search)
3. Save findings to a file (write_file)
4. Update its plan (write_todos again)

Nobody told it the exact sequence — the **system prompt** gave it a workflow, and the **ReAct loop** let it execute iteratively.

---

## Part 6: The 5 Pillars Preview

What we built today is just the beginning. The **Deep Agents from Scratch** series covers five pillars of agent development:

| Pillar | Topic | Session |
|--------|-------|---------|
| 1 | **Orchestration** — The ReAct loop (this notebook!) | Session 1 |
| 2 | **Memory** — Persisting context across interactions | Session 3 |
| 3 | **Tools** — Extending agent capabilities | Session 4 |
| 4 | **Agent Skills** — Composable, reusable behaviors | Session 4 |
| 5 | **Evaluation** — Measuring agent performance | Session 6 |

Each session builds on the last, taking you from a simple ReAct agent to a full-featured deep agent system.

## Next Steps

- **Session 2**: Deep dive into agent architecture — understanding graphs, nodes, and edges
- **Session 3**: Adding memory and context management
- **Session 4**: Building custom tools and agent skills

See you in the next session!